# ChurnSignal — Milestone 7: Model Explainability with SHAP
**Learning-First SaaS Customer Retention Intelligence**

Welcome to **Milestone 7**! Having crowned LightGBM as our Champion model in Milestone 6, we now open the "black box." A complex model that outputs a raw probability without explanation is practically useless to a Customer Success (CS) team. When a CS manager reaches out to an at-risk customer, they need to know *why* the customer is in danger.

---

### Key ML Learning Concept: Shapley Additive exPlanations (SHAP)
Rooted in cooperative game theory (Lloyd Shapley, Nobel Memorial Prize 1953; Lundberg & Lee 2017), SHAP conceptualizes each feature as a "player" in a coalition game where the "payout" is the model's prediction:

$$f(x) = \phi_0 + \sum_{j=1}^{M} \phi_j(x)$$

Where:
- $f(x)$ is the model's raw prediction (log-odds margin for tree classifiers)
- $\phi_0 = \mathbb{E}[f(X)]$ is the expected baseline value (population average log-odds)
- $\phi_j(x)$ is the local Shapley attribution for feature $j$ on instance $x$

### The 4 Mathematical Axioms of SHAP:
1. **Efficiency (Additivity)**: The sum of all feature attributions equals the difference between the prediction and the base expected value: $\sum \phi_j = f(x) - \phi_0$.
2. **Symmetry**: If two features contribute identically across all possible subsets of features, their attributions are equal.
3. **Dummy / Null Player**: A feature that has no impact on any subset prediction receives an attribution of exactly 0.
4. **Additivity / Linearity**: When combining models (ensembles), the Shapley value of the ensemble is the sum/average of the individual models' Shapley values.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add project root to sys.path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_prediction_eligible_data
from src.features.engineer import engineer_all_features
from src.features.pipeline import chronological_split, TARGET_COLUMN
from src.models.baselines import load_model
from src.explainability.explainer import (
    build_tree_explainer,
    compute_global_feature_importance,
    explain_single_customer,
    plot_shap_global_summary,
    plot_customer_waterfall
)

## 1. Load Preprocessed Data & Champion Artifacts

We load our pipeline and champion LightGBM model from `artifacts/` and prepare our validation partition.

In [ ]:
# 1. Load data and engineer features
eligible_df = load_prediction_eligible_data()
featured_df = engineer_all_features(eligible_df)
train_df, val_df, test_df = chronological_split(featured_df)

# 2. Load serialized pipeline & champion LightGBM
import joblib
pipeline = joblib.load(project_root / "artifacts" / "feature_pipeline.joblib")
champion_model = joblib.load(project_root / "artifacts" / "champion_model.joblib")

# 3. Transform validation features
feature_names = pipeline.get_feature_names()
X_val = pipeline.transform(val_df)
y_val = val_df[TARGET_COLUMN].to_numpy()

print(f"Validation cohort: {X_val.shape[0]:,} observations across {X_val.shape[1]} engineered features.")

## 2. Initialize TreeExplainer & Verify Additivity

TreeSHAP optimizes exact Shapley value computation from exponential time $\mathcal{O}(2^M)$ down to low-order polynomial time $\mathcal{O}(T L D^2)$, where $T$ is the number of trees, $L$ is maximum leaves, and $D$ is maximum tree depth.

Let's verify that the **Efficiency Axiom** holds to machine precision:

In [ ]:
# Build TreeExplainer
explainer = build_tree_explainer(champion_model)

# Verify additivity on a sample customer
sample_idx = 0
sample_x = X_val[sample_idx:sample_idx+1]
shap_sample = explainer(sample_x)

raw_margin = champion_model.predict(sample_x, raw_score=True)[0]
reconstructed_margin = explainer.expected_value + shap_sample.values[0].sum()

print(f"Base Value phi_0: {explainer.expected_value:.6f}")
print(f"Sum of feature SHAP values: {shap_sample.values[0].sum():.6f}")
print(f"Reconstructed Margin: {reconstructed_margin:.6f}")
print(f"Actual LightGBM raw score: {raw_margin:.6f}")
print(f"Absolute discrepancy: {abs(raw_margin - reconstructed_margin):.2e}")
assert np.isclose(raw_margin, reconstructed_margin, atol=1e-5), "Additivity failed!

## 3. Global Feature Attribution: Population-Level Signals

Traditional tree feature importance (Gini Impurity / Mean Decrease in Impurity) is notoriously biased toward high-cardinality and continuous features, and cannot indicate whether a feature pushes predictions up or down.

In contrast, Global SHAP measures the average absolute contribution across all validation records:
$$\text{Global Importance}_j = \frac{1}{N} \sum_{i=1}^{N} |\phi_j(x^{(i)})|$$

In [ ]:
# Compute global importance
importance_df = compute_global_feature_importance(explainer, X_val, feature_names)
print("Top 10 Global Features ranked by mean |SHAP| impact:")
display(importance_df.head(10))

# Display publication figure
display(Image(filename=str(project_root / "reports" / "figures" / "13_shap_global_summary.png")))

## 4. Local Feature Attribution: Explaining Individual Customers

Global importance tells CS leadership what to monitor at the macro level. But a customer success manager working with a specific account needs **local attribution**:
- Why is *this specific customer* predicted to churn?
- What are their primary risk drivers (pushing churn probability up)?
- What protective factors are currently keeping them retained (pulling churn probability down)?

Let's inspect the customer with the highest predicted churn probability in the validation cohort:

In [ ]:
# Generate probabilities for validation cohort
val_probs = champion_model.predict_proba(X_val)[:, 1]
top_risk_idx = np.argsort(val_probs)[::-1][0]
top_user_id = val_df.iloc[top_risk_idx]["user_id"]
top_user_prob = val_probs[top_risk_idx]
top_user_true = y_val[top_risk_idx]

print(f"Customer ID: {top_user_id}")
print(f"Predicted 30-Day Churn Probability: {top_user_prob:.2%}")
print(f"Actual Outcome: {'CHURNED' if top_user_true == 1 else 'RETAINED'}")

# Generate explanation
cust_vec = X_val[top_risk_idx]
explanation = explain_single_customer(explainer, cust_vec, feature_names, top_n=5)

print(f"\nBase Value (log-odds): {explanation['base_value']:.4f}")
print(f"Raw Score (sum + base): {explanation['raw_margin']:.4f}")

print("\nTop 5 Risk Drivers (Red Bars — Pushing Risk UP):")
for r in explanation["risk_drivers"]:
    print(f"  ▲ {r['Feature']}: SHAP={r['SHAP_Value']:+.4f} (Standardized Input={r['Feature_Input']:.3f})")

print("\nTop Protective Factors (Blue Bars — Pulling Risk DOWN):")
for p in explanation["protective_factors"]:
    print(f"  ▼ {p['Feature']}: SHAP={p['SHAP_Value']:+.4f} (Standardized Input={p['Feature_Input']:.3f})")

# Display Waterfall Plot
display(Image(filename=str(project_root / "reports" / "figures" / "14_customer_shap_waterfall.png")))

## 5. From Attribution to Action: Operational Playbooks

Notice how the local diagnosis transforms raw ML output into concrete Customer Success actions:

1. **`rolling_mean_usage_3m` & `feature_usage_score` are strongly negative (-1.86 and -1.50 SD)**:
   - *Signal*: Usage has plummeted over the last 90 days.
   - *Action Playbook*: Re-engagement onboarding campaign, product training session, or feature walkthrough.
2. **`support_tickets` is very high (+2.55 SD)**:
   - *Signal*: Friction or blocker in the software.
   - *Action Playbook*: Priority escalation to Senior Engineering / Solutions Architect to resolve open support tickets.
3. **`payment_failures` is elevated (+2.13 SD)**:
   - *Signal*: Dunning or credit card expiration friction.
   - *Action Playbook*: Automated billing reminder and payment update link before the next cycle.

SHAP provides the exact causal hypotheses that enable proactive retention rather than generic, wasted outreach!